# 🤖 Agentic RAG with Grok + LangGraph

**Agentic Retrieval-Augmented Generation** — a self-correcting pipeline that loops until it gets the answer right.

This notebook shows how to build a multi-step AI agent that:
1. **Retrieves** relevant documents from a vector database
2. **Generates** a draft answer using Grok (xAI)
3. **Evaluates** the draft for hallucinations and accuracy
4. **Loops** back to retrieval if the draft fails review

The orchestration is handled by **LangGraph** — a state machine framework that makes the agent loop easy to reason about and extend.

> 🔑 **Requires**: An [xAI API key](https://console.x.ai/) (free tier available)

> 🦃 *Created by Crunch, managed at [Copilotclaw/trainingdemo](https://github.com/Copilotclaw/trainingdemo)*

---

## 🏗️ Architecture

```
Question
   │
   ▼
┌─────────┐     ┌──────────┐     ┌──────────┐
│ Retrieve │────▶│ Generate │────▶│ Evaluate │
│  (RAG)  │◀────│ (Grok)   │     │  (Grok)  │
└─────────┘     └──────────┘     └──────────┘
    ▲                                  │
    │         FAIL (+ feedback)        │
    └──────────────────────────────────┘
                                       │
                    PASS               ▼
                               ✅ Final Answer
```

The **Evaluator** node uses Grok's structured output to return a strict JSON grade (`{is_accurate, feedback}`). If the draft fails, the feedback is appended to the next retrieval query — helping the agent find better context on the retry.

## Cell 1: Install Dependencies

We use `langchain-xai` for Grok integration, `chromadb` for the vector store, and `langgraph` for the state machine orchestration.

In [ ]:
# CELL 1: Setup & Installations
!pip install -qU langgraph langchain langchain-xai langchain-huggingface chromadb sentence-transformers pydantic

## Cell 2: API Keys & Grok Setup

Get your free API key at the [xAI Developer Console](https://console.x.ai/). We use `grok-2-latest` — it handles structured JSON output reliably, which is critical for the Evaluator node.

In [ ]:
# CELL 2: Environment Setup
import os
from getpass import getpass
from langchain_xai import ChatXAI

# BYOK: Enter your xAI API Key 
os.environ["XAI_API_KEY"] = getpass("Enter your xAI API Key: ")

# The "Brain" of our agent is now Grok!
llm = ChatXAI(
    model="grok-2-latest", 
    temperature=0
)

## Cell 3: The Vector Database

We embed 3 documents about a fictional server incident. The embeddings are generated locally using HuggingFace's `all-MiniLM-L6-v2` — **no API key needed for this part**, and it runs fine on Colab's free tier.

In [ ]:
# CELL 3: Initialize the Vector DB
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

docs = [
    Document(page_content="On March 12th, the server outage was caused by a DDoS attack on the EU-East region."),
    Document(page_content="Tuesday's 504 Gateway Timeout was traced back to a misconfigured API limit in the EU-West load balancer."),
    Document(page_content="To fix an API limit timeout, engineers must increase the concurrent connection threshold in the AWS console."),
]

vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"✅ Vector store ready with {len(docs)} documents")

## Cell 4: Define the Agent State

LangGraph passes a **state dictionary** between nodes. Think of it as a shared notepad that every agent step can read from and write to. The `loop_count` field is the safety valve — it prevents infinite retry loops.

In [ ]:
# CELL 4: Define the Agent State
from typing import TypedDict, List

class AgentState(TypedDict):
    question: str
    context: List[str]
    draft_answer: str
    feedback: str
    loop_count: int

## Cell 5: Define the Agent Nodes

Three nodes, each doing one job:

- **`retrieve_data`** — queries the vector store (appends evaluator feedback on retry)
- **`generate_draft`** — asks Grok to answer using only the retrieved context
- **`evaluate_draft`** — asks Grok to grade its own output as strict JSON (`{is_accurate, feedback}`)

LangChain's `with_structured_output(GraderOutput)` handles the JSON parsing automatically.

In [ ]:
# CELL 5: The Agentic Nodes
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

def retrieve_data(state: AgentState):
    print("-> AGENT: Retrieving Data...")
    search_query = state["question"] + " " + state.get("feedback", "")
    docs = retriever.invoke(search_query)
    context = [doc.page_content for doc in docs]
    return {"context": context, "loop_count": state.get("loop_count", 0) + 1}

def generate_draft(state: AgentState):
    print("-> AGENT: Generating Draft with Grok...")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a tech assistant. Answer the question using ONLY the context below.\n\nContext: {context}\n\nFeedback from Reviewer: {feedback}"),
        ("user", "{question}")
    ])
    chain = prompt | llm
    response = chain.invoke({"context": "\n".join(state["context"]), "question": state["question"], "feedback": state.get("feedback", "")})
    return {"draft_answer": response.content}

class GraderOutput(BaseModel):
    is_accurate: bool = Field(description="True if the draft perfectly answers the question using only context.")
    feedback: str = Field(description="If False, explain exactly what is missing or hallucinated.")

def evaluate_draft(state: AgentState):
    print("-> AGENT: Grok is Evaluating the Draft for Hallucinations...")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a strict QA reviewer. Does the Draft perfectly answer the Question based on the Context? If it hallucinates or misses key info, fail it and provide feedback."),
        ("user", "Question: {question}\nContext: {context}\nDraft: {draft_answer}")
    ])
    
    evaluator_llm = llm.with_structured_output(GraderOutput)
    chain = prompt | evaluator_llm
    
    result = chain.invoke({"question": state["question"], "context": "\n".join(state["context"]), "draft_answer": state["draft_answer"]})
    
    print(f"   [Evaluation Result: {'PASS' if result.is_accurate else 'FAIL'}]")
    if not result.is_accurate:
        print(f"   [Feedback: {result.feedback}]")
        
    return {"feedback": result.feedback if not result.is_accurate else "PASS"}

## Cell 6: Wire the Graph

LangGraph's `StateGraph` connects the nodes into a directed graph. The `routing_logic` function decides what happens after evaluation:

- **PASS** → done ✅
- **FAIL + loops < 3** → retry with feedback
- **loops ≥ 3** → bail out (max retries exceeded)

In [ ]:
# CELL 6: Build the State Machine Graph
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

workflow.add_node("Retrieve", retrieve_data)
workflow.add_node("Generate", generate_draft)
workflow.add_node("Evaluate", evaluate_draft)

workflow.add_edge("Retrieve", "Generate")
workflow.add_edge("Generate", "Evaluate")

def routing_logic(state: AgentState):
    if state["feedback"] == "PASS":
        return "End"
    elif state["loop_count"] >= 3:
        print("-> SYSTEM: Max loops reached. Exiting safely.")
        return "End"
    else:
        print("-> SYSTEM: Routing back to Retriever to try again.")
        return "Retrieve"

workflow.add_conditional_edges("Evaluate", routing_logic, {"Retrieve": "Retrieve", "End": END})
workflow.set_entry_point("Retrieve")
app = workflow.compile()

print("✅ Graph compiled successfully")

## Cell 7: Execute!

The question deliberately spans two documents: the *cause* of the timeout and the *region* where it happened. Watch how the agent retrieves, drafts, and self-evaluates.

In [ ]:
# CELL 7: Run the Scenario
inputs = {"question": "What caused Tuesday's 504 timeout and in which region did it happen?"}

print("=== STARTING AGENTIC RAG WITH GROK ===\n")
for output in app.stream(inputs):
    for key, value in output.items():
        pass 

print("\n=== FINAL VERIFIED OUTPUT ===")
# Get the final state
final_state = None
for output in app.stream(inputs):
    final_state = output

# Extract draft_answer from the last node output
if final_state:
    for key, value in final_state.items():
        if isinstance(value, dict) and "draft_answer" in value:
            print(value["draft_answer"])

## 🏗️ The Architectural Takeaway

Notice how the **orchestration logic** (the state machine, the loops, the retrieval mechanism) didn't care at all that we swapped the underlying model.

This is why investing in an orchestration framework like **LangGraph** is so critical. If a new, incredibly cheap, or vastly superior model drops next week, your enterprise pipeline doesn't break — **you just update Cell 2**.

### What makes this "Agentic"?

| Traditional RAG | Agentic RAG |
|----------------|-------------|
| Retrieve once → Generate once | Retrieve → Generate → Evaluate → loop |
| No self-correction | Feedback-driven retries |
| Single LLM call | LLM used for generation *and* quality control |
| Output is whatever came out | Output is verified before delivery |

### Next Steps
- **Swap the LLM**: Change `grok-2-latest` to any LangChain-supported model — same graph, same logic
- **Expand the knowledge base**: Replace the 3-doc demo with a real document corpus
- **Add more nodes**: Web search fallback, citation extraction, confidence scoring
- **Visualize the graph**: `app.get_graph().draw_mermaid()` prints a Mermaid diagram

> 🦃 *Built by Crunch — [Copilotclaw/trainingdemo](https://github.com/Copilotclaw/trainingdemo)*